In [ ]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null # Instalar SDK java 17

In [ ]:
!wget -q https://downloads.apache.org/spark/spark-4.0.2/spark-4.0.2-bin-hadoop3.tgz # Descargar Spark 4.0.1

In [ ]:
!tar xf spark-4.0.2-bin-hadoop3.tgz # Descomprimir la version de Spark

In [ ]:
!pip install -q findspark # Instalar la librería findspark

In [ ]:
!pip install -q pyspark # Instalar pyspark

## Funciones de fecha y hora

Al igual que el STRACT(DATE) y todas sus variantes o funciones de SQL similares pySpark nos permite trabajar con este tipo de funciones

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [7]:
data = [
    (1, "2023-10-01 12:30:00", "20231005", None, "A", [1, 2, 3]),
    (2, "2023-11-15 08:45:10", "20231120", "Promo", "B", [4, 5]),
    (3, None, None, "Cyber", None, []),
    (4, "2024-01-01 00:00:00", "20240101", None, "C", [10, 20, 30, 40])
]

schema = ["id", "fecha_str_1", "fecha_str_2", "evento", "categoria", "numeros"]
df = spark.createDataFrame(data, schema)
df.show()

+---+-------------------+-----------+------+---------+----------------+
| id|        fecha_str_1|fecha_str_2|evento|categoria|         numeros|
+---+-------------------+-----------+------+---------+----------------+
|  1|2023-10-01 12:30:00|   20231005|  NULL|        A|       [1, 2, 3]|
|  2|2023-11-15 08:45:10|   20231120| Promo|        B|          [4, 5]|
|  3|               NULL|       NULL| Cyber|     NULL|              []|
|  4|2024-01-01 00:00:00|   20240101|  NULL|        C|[10, 20, 30, 40]|
+---+-------------------+-----------+------+---------+----------------+



### col, to_timestamp, to_date, withColumn, otherwise

In [9]:
from pyspark.sql import functions as F

df_fechas = df.withColumn("fecha_dt", F.to_date(F.col("fecha_str_1"))) \
              .withColumn("timestamp_ts", F.to_timestamp(F.col("fecha_str_1"))) \
              .withColumn("fecha_formato_especial", F.to_date(F.col("fecha_str_2"), "yyyyMMdd")) \
              .withColumn("pais", F.lit("México"))

df_fechas.select("id", "fecha_dt", "timestamp_ts", "fecha_formato_especial", "pais").show()

+---+----------+-------------------+----------------------+------+
| id|  fecha_dt|       timestamp_ts|fecha_formato_especial|  pais|
+---+----------+-------------------+----------------------+------+
|  1|2023-10-01|2023-10-01 12:30:00|            2023-10-05|México|
|  2|2023-11-15|2023-11-15 08:45:10|            2023-11-20|México|
|  3|      NULL|               NULL|                  NULL|México|
|  4|2024-01-01|2024-01-01 00:00:00|            2024-01-01|México|
+---+----------+-------------------+----------------------+------+



In [10]:
df_logica = df.withColumn("tipo_categoria",
                          F.when(F.col("categoria") == "A", "Premium")
                           .when(F.col("categoria") == "B", "Estandar")
                           .otherwise("Sin Categoria")) \
              .withColumn("evento_final", F.coalesce(F.col("evento"), F.lit("Venta Regular")))

df_logica.select("id", "categoria", "tipo_categoria", "evento", "evento_final").show()

+---+---------+--------------+------+-------------+
| id|categoria|tipo_categoria|evento| evento_final|
+---+---------+--------------+------+-------------+
|  1|        A|       Premium|  NULL|Venta Regular|
|  2|        B|      Estandar| Promo|        Promo|
|  3|     NULL| Sin Categoria| Cyber|        Cyber|
|  4|        C| Sin Categoria|  NULL|Venta Regular|
+---+---------+--------------+------+-------------+



### datediff, months_between, last_day, to_date

In [11]:
from pyspark.sql import functions as F

# 1. Preparamos un DataFrame con fechas de inicio y fin
data_fechas = [
    (1, "2023-01-15", "2023-03-20"),
    (2, "2023-12-01", "2024-01-01"),
    (3, "2024-02-10", "2024-02-28"), # Año bisiesto
    (4, "2024-05-05", None)
]

df_periodos = spark.createDataFrame(data_fechas, ["id", "fecha_inicio", "fecha_fin"])

# Convertir strings a DateType para poder operar
df_periodos = df_periodos.withColumn("fecha_inicio", F.to_date("fecha_inicio")) \
                         .withColumn("fecha_fin", F.to_date("fecha_fin"))

# 2. Aplicar funciones de cálculo
df_calculos = df_periodos.withColumn("dias_diferencia", F.datediff(F.col("fecha_fin"), F.col("fecha_inicio"))) \
                         .withColumn("meses_diferencia", F.round(F.months_between(F.col("fecha_fin"), F.col("fecha_inicio")), 2)) \
                         .withColumn("ultimo_dia_mes_inicio", F.last_day(F.col("fecha_inicio"))) \
                         .withColumn("ultimo_dia_mes_fin", F.last_day(F.col("fecha_fin")))

df_calculos.select(
    "id",
    "fecha_inicio",
    "fecha_fin",
    "dias_diferencia",
    "meses_diferencia",
    "ultimo_dia_mes_inicio"
).show()

+---+------------+----------+---------------+----------------+---------------------+
| id|fecha_inicio| fecha_fin|dias_diferencia|meses_diferencia|ultimo_dia_mes_inicio|
+---+------------+----------+---------------+----------------+---------------------+
|  1|  2023-01-15|2023-03-20|             64|            2.16|           2023-01-31|
|  2|  2023-12-01|2024-01-01|             31|             1.0|           2023-12-31|
|  3|  2024-02-10|2024-02-28|             18|            0.58|           2024-02-29|
|  4|  2024-05-05|      NULL|           NULL|            NULL|           2024-05-31|
+---+------------+----------+---------------+----------------+---------------------+



###  date_sub, date_add, to_date

In [12]:
# Continuando con el DataFrame de periodos anterior o creando uno nuevo:
data_aritmetica = [
    (1, "2024-01-01", 7),  # Sumar una semana
    (2, "2024-05-15", 30), # Sumar un mes aproximado
    (3, "2024-12-31", 1),  # Salto de año
    (4, "2024-03-01", 1)   # Caso año bisiesto (restar 1 para ver febrero)
]

df_base = spark.createDataFrame(data_aritmetica, ["id", "fecha_referencia", "intervalo"])
df_base = df_base.withColumn("fecha_referencia", F.to_date("fecha_referencia"))

# Aplicando las transformaciones
df_aritmetica = df_base.withColumn("proxima_semana", F.date_add(F.col("fecha_referencia"), 7)) \
                       .withColumn("ayer", F.date_sub(F.col("fecha_referencia"), 1)) \
                       .withColumn("proyeccion_dinamica", F.date_add(F.col("fecha_referencia"), 15))

df_aritmetica.select(
    "fecha_referencia",
    "proxima_semana",
    "ayer",
    "proyeccion_dinamica"
).show()

+----------------+--------------+----------+-------------------+
|fecha_referencia|proxima_semana|      ayer|proyeccion_dinamica|
+----------------+--------------+----------+-------------------+
|      2024-01-01|    2024-01-08|2023-12-31|         2024-01-16|
|      2024-05-15|    2024-05-22|2024-05-14|         2024-05-30|
|      2024-12-31|    2025-01-07|2024-12-30|         2025-01-15|
|      2024-03-01|    2024-03-08|2024-02-29|         2024-03-16|
+----------------+--------------+----------+-------------------+



In [13]:
# Creamos un DataFrame con timestamps detallados
data_detallada = [
    (1, "2024-05-09 14:30:45"),
    (2, "2023-12-31 23:59:59"),
    (3, "2024-01-01 00:00:01")
]

df_tiempo = spark.createDataFrame(data_detallada, ["id", "evento_ts"])
df_tiempo = df_tiempo.withColumn("evento_ts", F.to_timestamp("evento_ts"))

# Extraer cada componente
df_componentes = df_tiempo.select(
    F.col("evento_ts"),
    F.year("evento_ts").alias("año"),
    F.month("evento_ts").alias("mes"),
    F.dayofmonth("evento_ts").alias("dia_mes"),
    F.dayofyear("evento_ts").alias("dia_año"),
    F.hour("evento_ts").alias("hora"),
    F.minute("evento_ts").alias("minuto"),
    F.second("evento_ts").alias("segundo")
)

df_componentes.show()

+-------------------+----+---+-------+-------+----+------+-------+
|          evento_ts| año|mes|dia_mes|dia_año|hora|minuto|segundo|
+-------------------+----+---+-------+-------+----+------+-------+
|2024-05-09 14:30:45|2024|  5|      9|    130|  14|    30|     45|
|2023-12-31 23:59:59|2023| 12|     31|    365|  23|    59|     59|
|2024-01-01 00:00:01|2024|  1|      1|      1|   0|     0|      1|
+-------------------+----+---+-------+-------+----+------+-------+



In [14]:
# Creamos datos con espacios intencionales
data_sucia = [
    (1, "  Juan Perez", "Espacios izquierda"),
    (2, "Maria Garcia  ", "Espacios derecha"),
    (3, "   Luis Lopez   ", "Ambos lados"),
    (4, "Ana Sanchez", "Sin espacios")
]

df_strings = spark.createDataFrame(data_sucia, ["id", "nombre", "descripcion"])

# Aplicamos las funciones de limpieza
df_limpio = df_strings.withColumn("ltrim", F.ltrim(F.col("nombre"))) \
                      .withColumn("rtrim", F.col("nombre")) \
                      .withColumn("trim", F.trim(F.col("nombre")))

# Para notar la diferencia en el resultado, podemos medir la longitud antes y después
df_verificacion = df_limpio.select(
    F.concat(F.lit("'"), F.col("nombre"), F.lit("'")).alias("original"),
    F.concat(F.lit("'"), F.col("trim"), F.lit("'")).alias("con_trim"),
    F.length(F.col("nombre")).alias("longitud_original"),
    F.length(F.col("trim")).alias("longitud_final")
)

df_verificacion.show()

+------------------+--------------+-----------------+--------------+
|          original|      con_trim|longitud_original|longitud_final|
+------------------+--------------+-----------------+--------------+
|    '  Juan Perez'|  'Juan Perez'|               12|            10|
|  'Maria Garcia  '|'Maria Garcia'|               14|            12|
|'   Luis Lopez   '|  'Luis Lopez'|               16|            10|
|     'Ana Sanchez'| 'Ana Sanchez'|               11|            11|
+------------------+--------------+-----------------+--------------+



### lpad, rpad

In [15]:
# Datos de ejemplo: IDs que deberían tener 5 dígitos y códigos de área
data_formato = [
    (1, "123", "MX"),
    (2, "45", "USA"),
    (3, "9", "CAN"),
    (4, "8888", "ARG")
]

df_format = spark.createDataFrame(data_formato, ["id_numerico", "codigo", "pais"])

# Transformaciones
# 1. Usamos col() para referenciar
# 2. lpad para estandarizar el código a 5 dígitos rellenando con "0"
# 3. rpad para que el país siempre ocupe 5 espacios rellenando con "."

df_final = df_format.select(
    F.col("id_numerico"), # Uso básico de col
    F.lpad(F.col("codigo"), 5, "0").alias("codigo_formateado"),
    F.rpad(F.col("pais"), 5, ".").alias("pais_formateado")
)

df_final.show()

+-----------+-----------------+---------------+
|id_numerico|codigo_formateado|pais_formateado|
+-----------+-----------------+---------------+
|          1|            00123|          MX...|
|          2|            00045|          USA..|
|          3|            00009|          CAN..|
|          4|            08888|          ARG..|
+-----------+-----------------+---------------+



### lower, upper, initcap, regexp_replace

In [16]:
# Datos de ejemplo con formatos inconsistentes
data_text = [
    ("001", "juan", "perez", "ID-555-2024"),
    ("002", "MARIA", "GARCIA", "ID-777-2023"),
    ("003", "luis", "loPeZ", "ID#999#2022")
]

df_text = spark.createDataFrame(data_text, ["id", "nombre", "apellido", "folio_sucio"])

# Aplicamos las transformaciones
df_text_final = df_text.select(
    # Unir nombre y apellido con un espacio
    F.concat_ws(" ", F.col("nombre"), F.col("apellido")).alias("nombre_completo"),

    # Estandarizar mayúsculas/minúsculas
    F.lower(F.col("nombre")).alias("minusc"),
    F.upper(F.col("apellido")).alias("MAYUSC"),
    F.initcap(F.col("nombre")).alias("Formato_Nombre"), # Primera letra mayúscula

    # Invertir cadena
    F.reverse(F.col("id")).alias("id_invertido"),

    # Limpiar el folio: reemplazar cualquier caracter que no sea número por un guion
    # El regex "[^0-9]+" busca todo lo que NO sea un dígito
    F.regexp_replace(F.col("folio_sucio"), "[^0-9]+", "-").alias("folio_limpio")
)

df_text_final.show()

+---------------+------+------+--------------+------------+------------+
|nombre_completo|minusc|MAYUSC|Formato_Nombre|id_invertido|folio_limpio|
+---------------+------+------+--------------+------------+------------+
|     juan perez|  juan| PEREZ|          Juan|         100|   -555-2024|
|   MARIA GARCIA| maria|GARCIA|         Maria|         200|   -777-2023|
|     luis loPeZ|  luis| LOPEZ|          Luis|         300|   -999-2022|
+---------------+------+------+--------------+------------+------------+



### to_json, from_json

In [18]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Definir el esquema de los datos que esperamos dentro del JSON
# Es obligatorio para que Spark sepa cómo interpretar el string
json_schema = StructType([
    StructField("ciudad", StringType(), True),
    StructField("codigo_postal", IntegerType(), True),
    StructField("clima", StringType(), True)
])

# 2. Datos de ejemplo: Una columna con un string que parece JSON
data_json = [
    (1, '{"ciudad": "CDMX", "codigo_postal": 11000, "clima": "Soleado"}'),
    (2, '{"ciudad": "Bogotá", "codigo_postal": 11011, "clima": "Lluvia"}'),
    (3, '{"ciudad": "Madrid", "codigo_postal": 28001, "clima": "Nublado"}')
]

df_raw = spark.createDataFrame(data_json, ["id", "datos_string"])

# 3. Aplicar from_json para convertir el string en un objeto (Struct)
df_parsed = df_raw.withColumn("datos_struct", F.from_json(F.col("datos_string"), json_schema))

# Ahora podemos acceder a los campos internos del struct con el punto (.)
df_final_json = df_parsed.select(
    "id",
    "datos_struct.ciudad",
    "datos_struct.clima"
)

df_final_json.show()

# 4. Aplicar to_json para volver a convertir un objeto en string
# Útil para enviar datos de vuelta a una API o Kafka
df_back_to_string = df_final_json.withColumn("json_empaquetado", F.to_json(F.struct("ciudad", "clima")))

df_back_to_string.select("id", "json_empaquetado").show(truncate=False)

+---+------+-------+
| id|ciudad|  clima|
+---+------+-------+
|  1|  CDMX|Soleado|
|  2|Bogotá| Lluvia|
|  3|Madrid|Nublado|
+---+------+-------+

+---+-------------------------------------+
|id |json_empaquetado                     |
+---+-------------------------------------+
|1  |{"ciudad":"CDMX","clima":"Soleado"}  |
|2  |{"ciudad":"Bogotá","clima":"Lluvia"} |
|3  |{"ciudad":"Madrid","clima":"Nublado"}|
+---+-------------------------------------+

